In [ ]:
# CELDA 1 — Setup: clustering mejorado con HDBSCAN + PSI monitoring
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, joblib, time, warnings
warnings.filterwarnings('ignore')

# Instalar HDBSCAN si no esta
!pip install -q hdbscan

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
import hdbscan

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

PROJECT_PATH   = '/content/drive/MyDrive/HortifrutCostosImport'
PROCESSED_PATH = f'{PROJECT_PATH}/data/processed'
MODELS_PATH    = f'{PROJECT_PATH}/models'
FIGURES_PATH   = f'{PROJECT_PATH}/reports/figures'

# CORRECCIÓN: cargar los datasets POST feature-engineering (tienen Proveedor_norm,
# incoterm_familia, etc.). Concatenamos train + test para tener TODAS las operaciones.
train_fe = pd.read_parquet(f'{PROCESSED_PATH}/dataset_modelable_train.parquet')
test_fe  = pd.read_parquet(f'{PROCESSED_PATH}/dataset_modelable_test.parquet')
df = pd.concat([train_fe, test_fe], ignore_index=True)
df = df[df['Importe_Total_PEN'] > 0].copy()

print(f'Dataset post-FE (train+test): {df.shape}')
print(f'Operaciones unicas: {df["Nro. Ope."].nunique():,}')
print(f'Columnas disponibles: {sorted(df.columns.tolist())}')


In [ ]:
# CELDA 2 — Agregar a nivel operacion (un registro por despacho)
def modo(series):
    s = series.dropna().astype(str)
    return s.mode().iloc[0] if len(s) > 0 else 'DESCONOCIDO'

# Construir el dict de agregacion de forma defensiva:
# solo incluir columnas que existen en el dataframe
agg_dict = {
    'peso_total':        ('Peso Bruto (kg)',           'max'),
    'contenedores':      ('Cantidad de Contenedores',  'max'),
    'bultos':            ('Cantidad de Bultos (BULKS)', 'sum'),
    'costo_total_pen':   ('Importe_Total_PEN',          'sum'),
    'n_facturas':        ('Importe_Total_PEN',          'count'),
    'n_conceptos':       ('Concepto Canónico',          'nunique'),
    'modalidad':         ('Modalidad (MODE Y TYPE)',    modo),
    'incoterm':          ('incoterm_familia',           modo),
}

# Columnas opcionales: agregar solo si existen
if 'Proveedor_norm' in df.columns:
    agg_dict['n_proveedores'] = ('Proveedor_norm', 'nunique')
else:
    agg_dict['n_proveedores'] = ('Proveedor', 'nunique')

if 'Proveedor Principal_norm' in df.columns:
    agg_dict['proveedor_principal'] = ('Proveedor Principal_norm', modo)
elif 'Proveedor Principal' in df.columns:
    agg_dict['proveedor_principal'] = ('Proveedor Principal', modo)

# POL y POD: buscar la columna correcta
col_pol = next((c for c in df.columns if c.upper() == 'POL'), None)
col_pod = next((c for c in df.columns if c.upper() == 'POD'), None)
if col_pol:
    agg_dict['pol'] = (col_pol, modo)
if col_pod:
    agg_dict['pod'] = (col_pod, modo)

df_ops = df.groupby('Nro. Ope.').agg(**agg_dict).reset_index()

# Garantizar que pol/pod siempre existan (aunque sea como placeholder)
if 'pol' not in df_ops.columns:
    df_ops['pol'] = 'DESCONOCIDO'
if 'pod' not in df_ops.columns:
    df_ops['pod'] = 'DESCONOCIDO'
if 'proveedor_principal' not in df_ops.columns:
    df_ops['proveedor_principal'] = 'DESCONOCIDO'

print(f'Operaciones: {len(df_ops)}')
print(f'Columnas: {df_ops.columns.tolist()}')
print(df_ops[['costo_total_pen', 'n_facturas', 'n_conceptos', 'contenedores', 'bultos']].describe().round(1).to_string())


In [ ]:
# CELDA 3 — Preprocesamiento: escalar numericas + one-hot categoricas
NUM_OPS = ['peso_total', 'contenedores', 'bultos', 'costo_total_pen',
           'n_facturas', 'n_conceptos', 'n_proveedores']
CAT_OPS = ['modalidad', 'incoterm', 'pol', 'pod', 'proveedor_principal']

for col in NUM_OPS:
    df_ops[col] = pd.to_numeric(df_ops[col], errors='coerce')

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), NUM_OPS),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_OPS)
])

# Imputar nulos numericos antes
imp = SimpleImputer(strategy='median')
df_ops[NUM_OPS] = imp.fit_transform(df_ops[NUM_OPS])
for col in CAT_OPS:
    df_ops[col] = df_ops[col].fillna('DESCONOCIDO').astype(str)

X = preprocessor.fit_transform(df_ops)
print(f'Matriz de features para clustering: {X.shape}')

# Reduccion PCA a 15 componentes (mejor para HDBSCAN que espacio de alta dimension)
pca = PCA(n_components=15, random_state=42)
X_pca = pca.fit_transform(X)
varianza_acum = pca.explained_variance_ratio_.cumsum()[-1]
print(f'PCA a 15 componentes: {varianza_acum*100:.1f}% de varianza explicada')

# PCA 2D para visualizacion
pca2d = PCA(n_components=2, random_state=42)
X_2d = pca2d.fit_transform(X)


## Modelo 1: K-Means (referencia — mismo que version anterior)

In [ ]:
# CELDA 4 — K-Means K=6 (referencia para comparar con HDBSCAN)
K_FINAL = 6
kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=20)
df_ops['cluster_kmeans'] = kmeans.fit_predict(X_pca)

sil_km = silhouette_score(X_pca, df_ops['cluster_kmeans'], random_state=42)
dbi_km = davies_bouldin_score(X_pca, df_ops['cluster_kmeans'])
print(f'K-Means K=6: silhouette={sil_km:.3f}, DBI={dbi_km:.3f}')
print('Distribucion:', df_ops['cluster_kmeans'].value_counts().sort_index().to_dict())


## Modelo 2 (Nuevo): HDBSCAN — Clustering Jerárquico por Densidad

HDBSCAN es superior a K-Means para datos mixtos porque:
- No asume clusters esféricos
- Detecta outliers automáticamente (cluster -1)
- No requiere especificar K de antemano
- Produce probabilidades de pertenencia suaves
- Más robusto a la escala y forma de los clusters

In [ ]:
# CELDA 5 — HDBSCAN: busqueda del min_cluster_size optimo
print('Buscando parametros optimos para HDBSCAN...')

resultados_hdb = []
for min_cs in [10, 15, 20, 25, 30, 40]:
    for min_s in [3, 5, 8]:
        hdb = hdbscan.HDBSCAN(min_cluster_size=min_cs, min_samples=min_s,
                              metric='euclidean', cluster_selection_method='eom')
        labels = hdb.fit_predict(X_pca)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_outliers = (labels == -1).sum()
        pct_outliers = n_outliers / len(labels) * 100

        if n_clusters >= 3:
            sil = silhouette_score(X_pca, labels, sample_size=min(500, len(labels)),
                                   random_state=42) if n_clusters > 1 else -1
            resultados_hdb.append({
                'min_cluster_size': min_cs, 'min_samples': min_s,
                'n_clusters': n_clusters, 'n_outliers': n_outliers,
                'pct_outliers': round(pct_outliers, 1), 'silhouette': round(sil, 3)
            })
            print(f'  min_cs={min_cs:2d} min_s={min_s}: clusters={n_clusters} '
                  f'outliers={n_outliers}({pct_outliers:.1f}%) sil={sil:.3f}')

# Seleccionar mejor configuracion (max silhouette, outliers < 10%)
df_hdb_res = pd.DataFrame(resultados_hdb)
df_hdb_res = df_hdb_res[df_hdb_res['pct_outliers'] < 10]
if len(df_hdb_res) > 0:
    best_row = df_hdb_res.loc[df_hdb_res['silhouette'].idxmax()]
    BEST_MIN_CS = int(best_row['min_cluster_size'])
    BEST_MIN_S  = int(best_row['min_samples'])
    print(f'\nMejor config: min_cluster_size={BEST_MIN_CS}, min_samples={BEST_MIN_S}')
else:
    BEST_MIN_CS, BEST_MIN_S = 20, 5
    print('\nUsando parametros por defecto: min_cluster_size=20, min_samples=5')


In [ ]:
# CELDA 6 — HDBSCAN final con parametros optimos
t0 = time.time()
hdb_final = hdbscan.HDBSCAN(
    min_cluster_size=BEST_MIN_CS,
    min_samples=BEST_MIN_S,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True  # necesario para predecir nuevos puntos
)
df_ops['cluster_hdbscan'] = hdb_final.fit_predict(X_pca)
df_ops['hdbscan_proba']   = hdb_final.probabilities_  # confianza de pertenencia
print(f'HDBSCAN entrenado en {time.time()-t0:.1f}s')

n_clusters_hdb = len(set(df_ops['cluster_hdbscan'].unique())) - (1 if -1 in df_ops['cluster_hdbscan'].values else 0)
n_outliers_hdb = (df_ops['cluster_hdbscan'] == -1).sum()

mask_no_outlier = df_ops['cluster_hdbscan'] != -1
sil_hdb = silhouette_score(X_pca[mask_no_outlier], df_ops.loc[mask_no_outlier, 'cluster_hdbscan'],
                            random_state=42) if mask_no_outlier.sum() > 10 else 0

print(f'\nHDBSCAN resultados:')
print(f'  Clusters encontrados: {n_clusters_hdb}')
print(f'  Outliers: {n_outliers_hdb} ({n_outliers_hdb/len(df_ops)*100:.1f}%)')
print(f'  Silhouette (sin outliers): {sil_hdb:.3f}')
print(f'  K-Means silhouette:        {sil_km:.3f}')
print(f'\nDistribucion HDBSCAN:')
print(df_ops['cluster_hdbscan'].value_counts().sort_index().to_string())


In [ ]:
# CELDA 7 — Comparacion visual K-Means vs HDBSCAN
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# K-Means
for c in sorted(df_ops['cluster_kmeans'].unique()):
    mask = df_ops['cluster_kmeans'] == c
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1], s=15, alpha=0.6, label=f'C{c}')
axes[0].set_title(f'K-Means K=6\nSilhouette={sil_km:.3f}')
axes[0].legend(markerscale=2)

# HDBSCAN
clusters_hdb = sorted(df_ops['cluster_hdbscan'].unique())
cmap = plt.cm.tab10
for i, c in enumerate(clusters_hdb):
    mask = df_ops['cluster_hdbscan'] == c
    color = 'gray' if c == -1 else cmap(i / max(len(clusters_hdb), 1))
    label = 'Outlier' if c == -1 else f'C{c}'
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1], s=15, alpha=0.6,
                    color=color, label=label)
axes[1].set_title(f'HDBSCAN ({n_clusters_hdb} clusters + outliers)\nSilhouette={sil_hdb:.3f}')
axes[1].legend(markerscale=2, ncol=2)

for ax in axes:
    ax.set_xlabel('PCA 1'); ax.set_ylabel('PCA 2')

plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}/26_hdbscan_vs_kmeans.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# CELDA 8 — Caracterizacion de perfiles HDBSCAN
print('Perfiles operativos HDBSCAN:')
print('=' * 80)

for c in sorted(df_ops['cluster_hdbscan'].unique()):
    sub = df_ops[df_ops['cluster_hdbscan'] == c]
    pct = len(sub) / len(df_ops) * 100
    print(f'\nCluster {c} ({'OUTLIERS' if c == -1 else 'perfil'}) — {len(sub)} ops ({pct:.1f}%)')
    print(f'  Costo total PEN: mediana=S/{sub["costo_total_pen"].median():,.0f} max=S/{sub["costo_total_pen"].max():,.0f}')
    print(f'  Facturas/op:     mediana={sub["n_facturas"].median():.1f}')
    print(f'  Conceptos/op:    mediana={sub["n_conceptos"].median():.1f}')
    if 'contenedores' in sub.columns:
        print(f'  Contenedores:    mediana={sub["contenedores"].median():.1f}')
    print(f'  Modalidad top:   {sub["modalidad"].value_counts().index[0] if len(sub) > 0 else "N/A"}')
    print(f'  Proveedor top:   {sub["proveedor_principal"].value_counts().index[0] if len(sub) > 0 else "N/A"}')
    print(f'  Proba media:     {sub["hdbscan_proba"].mean():.2f}  (1=certeza total)')


## Mejora P3-11: PSI (Population Stability Index) — Monitoreo de Drift

El PSI mide si la distribución de una variable cambia significativamente entre el conjunto de entrenamiento y producción. Permite detectar drift antes de que el modelo falle.

**Interpretación:** PSI < 0.1 = estable, 0.1-0.25 = cambio moderado, > 0.25 = drift crítico

In [ ]:
# CELDA 9 — Calculo de PSI entre train y test
train_all = pd.read_parquet(f'{PROCESSED_PATH}/dataset_modelable_train.parquet')
test_all  = pd.read_parquet(f'{PROCESSED_PATH}/dataset_modelable_test.parquet')
train_all = train_all[train_all['Importe_Total_PEN'] > 0]
test_all  = test_all[test_all['Importe_Total_PEN'] > 0]

def calcular_psi_numerico(x_base, x_prod, n_bins=10):
    """PSI para variables numericas."""
    # Definir bins usando la distribucion del train
    bins = np.percentile(x_base.dropna(), np.linspace(0, 100, n_bins + 1))
    bins = np.unique(bins)
    if len(bins) < 3:
        return 0.0
    bins[0]  = -np.inf
    bins[-1] = np.inf

    base_counts = np.histogram(x_base, bins=bins)[0] + 1e-6
    prod_counts = np.histogram(x_prod, bins=bins)[0] + 1e-6
    base_pct = base_counts / base_counts.sum()
    prod_pct = prod_counts / prod_counts.sum()

    psi = np.sum((prod_pct - base_pct) * np.log(prod_pct / base_pct))
    return round(psi, 4)

def calcular_psi_categorico(x_base, x_prod):
    """PSI para variables categoricas."""
    cats = set(x_base.dropna().unique()) | set(x_prod.dropna().unique())
    base_pct = x_base.value_counts(normalize=True)
    prod_pct = x_prod.value_counts(normalize=True)

    psi = 0
    for c in cats:
        b = base_pct.get(c, 1e-6)
        p = prod_pct.get(c, 1e-6)
        psi += (p - b) * np.log(p / b)
    return round(psi, 4)

# Calcular PSI para las features clave
print('PSI: Estabilidad de features entre TRAIN (2020-2024) y TEST (2025-2026)')
print('=' * 70)
print(f'  PSI < 0.10  = ESTABLE (verde)')
print(f'  PSI 0.10-0.25 = CAMBIO MODERADO (amarillo)')
print(f'  PSI > 0.25  = DRIFT CRITICO (rojo)')
print()

COLS_NUMERICAS_PSI = [
    'Importe_Total_PEN', 'tarifa_historica', 'dias_desde_inicio',
    'Cantidad de Contenedores', 'Cantidad de Bultos (BULKS)',
]
COLS_CAT_PSI = [
    'Concepto Canónico', 'incoterm_familia', 'Modalidad (MODE Y TYPE)',
]

resultados_psi = []

for col in COLS_NUMERICAS_PSI:
    if col not in train_all.columns:
        continue
    psi_val = calcular_psi_numerico(train_all[col], test_all[col])
    estado = 'ESTABLE' if psi_val < 0.10 else ('MODERADO' if psi_val < 0.25 else 'DRIFT!')
    resultados_psi.append({'feature': col, 'tipo': 'num', 'PSI': psi_val, 'estado': estado})
    print(f'  {col:<40} PSI={psi_val:.4f}  {estado}')

for col in COLS_CAT_PSI:
    if col not in train_all.columns:
        continue
    psi_val = calcular_psi_categorico(train_all[col], test_all[col])
    estado = 'ESTABLE' if psi_val < 0.10 else ('MODERADO' if psi_val < 0.25 else 'DRIFT!')
    resultados_psi.append({'feature': col, 'tipo': 'cat', 'PSI': psi_val, 'estado': estado})
    print(f'  {col:<40} PSI={psi_val:.4f}  {estado}')

df_psi = pd.DataFrame(resultados_psi).sort_values('PSI', ascending=False)
print(f'\nFeatures con DRIFT CRITICO (PSI > 0.25): {(df_psi["PSI"] > 0.25).sum()}')
print(f'Features con cambio MODERADO (0.10-0.25): {((df_psi["PSI"] >= 0.10) & (df_psi["PSI"] <= 0.25)).sum()}')


In [ ]:
# CELDA 10 — Guardar modelos de clustering y PSI
os.makedirs(MODELS_PATH, exist_ok=True)

joblib.dump(kmeans,      f'{MODELS_PATH}/kmeans_k6.joblib')
joblib.dump(hdb_final,   f'{MODELS_PATH}/hdbscan_model.joblib')
joblib.dump(preprocessor,f'{MODELS_PATH}/clustering_preprocessor.joblib')
joblib.dump(pca,         f'{MODELS_PATH}/clustering_pca15.joblib')
joblib.dump(df_psi,      f'{MODELS_PATH}/psi_baseline.joblib')

# Guardar operaciones con clusters
df_ops.to_parquet(f'{PROCESSED_PATH}/operaciones_con_clusters_v2.parquet', index=False)

# Guardar tabla PSI como CSV para referencia
df_psi.to_csv(f'{PROJECT_PATH}/reports/psi_baseline_2024_vs_2025.csv', index=False)

for fn in ['kmeans_k6.joblib', 'hdbscan_model.joblib',
           'clustering_preprocessor.joblib', 'clustering_pca15.joblib']:
    sz = os.path.getsize(f'{MODELS_PATH}/{fn}') / 1024 if os.path.exists(f'{MODELS_PATH}/{fn}') else 0
    print(f'  {fn}: {sz:.1f} KB')

print('\n=== RESUMEN FINAL TRACK CLUSTERING ===')
print(f'K-Means K=6:  silhouette={sil_km:.3f}')
print(f'HDBSCAN:      silhouette={sil_hdb:.3f} ({n_clusters_hdb} clusters + {n_outliers_hdb} outliers)')
print(f'PSI baselines calculados para {len(df_psi)} features')
print('Para monitoreo: ejecutar calcular_psi_* mensualmente con datos de produccion')
